# Customer Churn Prediction Using Data Science and Machine Learning 📊
**Academic Lab Project | B.Tech Data Science & AI**

---
### 📌 Problem Statement
Customer churn represents the percentage of customers who cease doing business with a company. In subscription telecom businesses, retaining an existing customer costs 5x to 7x less than acquiring a new one.

In this project, we build a complete Machine Learning pipeline using **Logistic Regression** on the **IBM Telco Customer Churn** dataset.

In [ ]:
# 1. Import Essential Libraries
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import joblib

# Visual styling
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (9, 5)
print('Libraries successfully imported!')

## Step 1: Data Loading & Initial Inspection
We load the `Telco-Customer-Churn.csv` dataset and inspect its structure, dimensions, and data types.

In [ ]:
data_path = 'data/Telco-Customer-Churn.csv'
df = pd.read_csv(data_path)
print('Dataset Shape:', df.shape)
df.head()

## Step 2: Data Cleaning & Preprocessing
Real-world telecom data often contains anomalies:
1. `TotalCharges` is stored as an `object` (string) because 11 rows contain blank whitespace `' '` where tenure is 0.
2. We convert `TotalCharges` to `float64` and impute `0.0` for 0-tenure customers.
3. `customerID` is an arbitrary identifier and must be dropped.
4. Target `Churn` is mapped to binary integers: `Yes` -> 1, `No` -> 0.

In [ ]:
df_clean = df.copy()

# Convert TotalCharges with coercion
df_clean['TotalCharges'] = pd.to_numeric(df_clean['TotalCharges'], errors='coerce').fillna(0.0)

# Drop customerID
if 'customerID' in df_clean.columns:
    df_clean.drop(columns=['customerID'], inplace=True)

# Map target Churn
df_clean['Churn'] = df_clean['Churn'].map({'Yes': 1, 'No': 0})

print('Data cleaning complete!')
df_clean.info()

## Step 3: Exploratory Data Analysis (EDA)
We visualize key variables to extract actionable business insights.

In [ ]:
# 3.1 Churn Distribution (Class Imbalance)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
counts = df_clean['Churn'].value_counts().sort_index()
labels = ['Retained (0)', 'Churned (1)']

sns.barplot(x=labels, y=counts.values, hue=labels, ax=axes[0], palette=['#2ecc71', '#e74c3c'], legend=False)
axes[0].set_title('Customer Churn Count', fontweight='bold')

axes[1].pie(counts.values, labels=labels, autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'], explode=(0, 0.08), startangle=90)
axes[1].set_title('Customer Churn Ratio', fontweight='bold')
plt.show()

In [ ]:
# 3.2 Churn by Contract Type
plt.figure(figsize=(8, 5))
contract_df = df_clean.copy()
contract_df['Status'] = contract_df['Churn'].map({1: 'Churned', 0: 'Retained'})
sns.countplot(data=contract_df, x='Contract', hue='Status', palette={'Retained': '#2ecc71', 'Churned': '#e74c3c'})
plt.title('Churn Distribution by Contract Type', fontweight='bold')
plt.show()

In [ ]:
# 3.3 Monthly Charges Density
plt.figure(figsize=(8, 5))
sns.kdeplot(data=df_clean[df_clean['Churn'] == 0]['MonthlyCharges'], label='Retained', color='#2ecc71', fill=True, alpha=0.3)
sns.kdeplot(data=df_clean[df_clean['Churn'] == 1]['MonthlyCharges'], label='Churned', color='#e74c3c', fill=True, alpha=0.3)
plt.title('Monthly Charges Distribution for Churned vs Retained Customers', fontweight='bold')
plt.xlabel('Monthly Charges ($)')
plt.legend()
plt.show()

In [ ]:
# 3.4 Tenure Distribution
plt.figure(figsize=(8, 5))
sns.histplot(data=contract_df, x='tenure', hue='Status', multiple='stack', bins=30, palette={'Retained': '#2ecc71', 'Churned': '#e74c3c'})
plt.title('Tenure Distribution (Months with Company)', fontweight='bold')
plt.xlabel('Tenure (Months)')
plt.show()

## Step 4: Machine Learning Pipeline & Train-Test Split
We separate features X and label y, perform a stratified 80/20 train/test split, and set up preprocessing.

In [ ]:
X = df_clean.drop(columns=['Churn'])
y = df_clean['Churn']

num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
cat_cols = [col for col in X.columns if col not in num_cols]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print(f'Training samples: {len(X_train)}, Testing samples: {len(X_test)}')

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols)
    ]
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

pipeline.fit(X_train, y_train)
print('Pipeline training complete!')

## Step 5: Model Evaluation on Unseen Test Set
We compute Accuracy, Precision, Recall, F1-Score, and plot the Confusion Matrix.

In [ ]:
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print(f'Accuracy : {acc * 100:.2f}%')
print(f'Precision: {prec * 100:.2f}%')
print(f'Recall   : {rec * 100:.2f}%')
print(f'F1-Score : {f1 * 100:.2f}%')
print('\nClassification Report:\n', classification_report(y_test, y_pred))

# Plot Confusion Matrix
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Pred Stay', 'Pred Churn'], yticklabels=['True Stay', 'True Churn'])
plt.title('Confusion Matrix')
plt.show()

## Step 6: Interpretability & Logistic Regression Coefficients
Logistic regression provides explainability through learned weights. Positive weights increase churn likelihood; negative weights encourage retention.

In [ ]:
ohe_cols = pipeline.named_steps['preprocessor'].named_transformers_['cat'].get_feature_names_out(cat_cols).tolist()
feature_names = num_cols + ohe_cols
weights = pipeline.named_steps['classifier'].coef_[0]

coef_df = pd.DataFrame({'Feature': feature_names, 'Weight': weights}).sort_values(by='Weight', key=abs, ascending=False)
coef_df.head(10)

## Step 7: Model Export for Streamlit Deployment
We serialize the pipeline using `joblib` so `app.py` can load it instantly.

In [ ]:
os.makedirs('models', exist_ok=True)
joblib.dump(pipeline, 'models/churn_pipeline.joblib')
print('Model saved to models/churn_pipeline.joblib!')